In [ ]:
from pathlib import Path
import os
import subprocess

ROOT = Path("/content")
REPO = ROOT / "multi-source-diffusion-models"
MAMBA = ROOT / "bin" / "micromamba"

os.environ["MAMBA_ROOT_PREFIX"] = str(ROOT / "micromamba")

def run(cmd, cwd=None):
    print("\n$", " ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), cwd=cwd, check=True)

run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-y", "-qq", "ffmpeg", "libsndfile1", "git", "curl", "bzip2"])

if not MAMBA.exists():
    run([
        "bash", "-lc",
        "curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba",
    ])

if not REPO.exists():
    run(["git", "clone", "https://github.com/gladia-research-group/multi-source-diffusion-models.git"], cwd=ROOT)

print("Micromamba:", MAMBA.exists(), MAMBA)
print("Repo:", REPO.exists(), REPO)

In [ ]:
from pathlib import Path
import os
import subprocess

ROOT = Path("/content")
REPO = ROOT / "multi-source-diffusion-models"
MAMBA = ROOT / "bin" / "micromamba"
ENV_NAME = "msdm"
ENV_PREFIX = ROOT / "micromamba" / "envs" / ENV_NAME
ENV_LIB = ENV_PREFIX / "lib"

os.environ["MAMBA_ROOT_PREFIX"] = str(ROOT / "micromamba")

def msdm_env():
    env = os.environ.copy()
    env["MAMBA_ROOT_PREFIX"] = str(ROOT / "micromamba")

    if ENV_LIB.exists():
        env["LD_LIBRARY_PATH"] = f"{ENV_LIB}:{env.get('LD_LIBRARY_PATH', '')}"
        env["LD_PRELOAD"] = str(ENV_LIB / "libstdc++.so.6")

    return env

def run_live(cmd, cwd=None):
    print("\n$", " ".join(map(str, cmd)))
    process = subprocess.Popen(
        list(map(str, cmd)),
        cwd=cwd,
        env=msdm_env(),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    lines = []
    for line in process.stdout:
        print(line, end="")
        lines.append(line)

    code = process.wait()
    if code != 0:
        print("\nLAST OUTPUT:")
        print("".join(lines[-120:]))
        raise RuntimeError(f"Command failed with code {code}")

if not ENV_PREFIX.exists():
    run_live([
        MAMBA, "create", "-y", "-n", ENV_NAME,
        "-c", "conda-forge",
        "python=3.9",
        "pip<24.1",
        "setuptools<70",
        "wheel",
        "numpy=1.26.4",
        "av=10.0.0",
        "ffmpeg",
        "libsndfile",
        "libstdcxx-ng",
        "libgcc-ng",
    ])

    run_live([
        MAMBA, "run", "-n", ENV_NAME,
        "python", "-m", "pip", "install",
        "--no-cache-dir",
        "-r", REPO / "requirements.txt",
    ])

    run_live([
        MAMBA, "run", "-n", ENV_NAME,
        "python", "-m", "pip", "uninstall", "-y",
        "torch", "torchaudio", "torchvision",
    ])

    run_live([
        MAMBA, "run", "-n", ENV_NAME,
        "python", "-m", "pip", "install",
        "--no-cache-dir",
        "--force-reinstall",
        "--extra-index-url", "https://download.pytorch.org/whl/cu113",
        "torch==1.12.1+cu113",
        "torchaudio==0.12.1+cu113",
        "torchvision==0.13.1+cu113",
    ])

    run_live([
        MAMBA, "run", "-n", ENV_NAME,
        "python", "-m", "pip", "install",
        "--no-cache-dir",
        "--force-reinstall",
        "--no-deps",
        "numpy==1.26.4",
        "torchmetrics==0.9.3",
        "protobuf==3.20.3",
        "setuptools==69.5.1",
        "gdown",
        "tqdm",
    ])
else:
    print("Using existing MSDM environment:", ENV_PREFIX)

run_live([
    MAMBA, "run", "-n", ENV_NAME,
    "python", "-c",
    """
import numpy
import torch
import torchaudio
import pytorch_lightning
import torchmetrics
import av

print("numpy:", numpy.__version__)
print("torch:", torch.__version__)
print("torchaudio:", torchaudio.__version__)
print("pytorch_lightning:", pytorch_lightning.__version__)
print("torchmetrics:", torchmetrics.__version__)
print("av:", av.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda version:", torch.version.cuda)
""",
])

In [ ]:
from pathlib import Path
import os
import subprocess
import tarfile

ROOT = Path("/content")
REPO = ROOT / "multi-source-diffusion-models"
MAMBA = ROOT / "bin" / "micromamba"
ENV_NAME = "msdm"

CKPT_DIR = REPO / "ckpts"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

os.environ["MAMBA_ROOT_PREFIX"] = str(ROOT / "micromamba")

ckpt_archive = CKPT_DIR / "msdm.tar"

def run(cmd):
    print("\n$", " ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), check=True)

def download_if_missing(url, out_path, min_size):
    if out_path.exists() and out_path.stat().st_size > min_size:
        print("Already downloaded:", out_path)
        return

    if out_path.exists():
        out_path.unlink()

    run([
        MAMBA, "run", "-n", ENV_NAME,
        "python", "-m", "gdown",
        "--fuzzy",
        url,
        "-O",
        out_path,
    ])

download_if_missing(
    "https://drive.google.com/file/d/1mfozibogvNrUaeS283OBz26MWNeZv-OO/view?usp=share_link",
    ckpt_archive,
    1_000_000_000,
)

if not sorted(CKPT_DIR.rglob("*.ckpt")):
    print("Extracting checkpoint archive...")
    with tarfile.open(ckpt_archive, mode="r:*") as tar:
        tar.extractall(CKPT_DIR)
else:
    print("Checkpoint already extracted.")

ckpts = sorted(CKPT_DIR.rglob("*.ckpt"))

print("ckpts:")
for path in ckpts:
    print(" ", path)

if not ckpts:
    raise RuntimeError("No .ckpt found.")

In [ ]:
from pathlib import Path

ROOT = Path("/content")
REPO = ROOT / "multi-source-diffusion-models"

N_GENERATIONS = 64
GEN_BATCH_SIZE = 16
NUM_STEPS = 150
SAMPLE_LEN = 262144
S_CHURN = 20.0
SEED = 42

gen_dir = REPO / "output" / "generation" / f"msdm-total-generation-{N_GENERATIONS}samples-{NUM_STEPS}steps"
gen_dir.mkdir(parents=True, exist_ok=True)

print("Output folder:", gen_dir)
print("N_GENERATIONS:", N_GENERATIONS)
print("GEN_BATCH_SIZE:", GEN_BATCH_SIZE)
print("NUM_STEPS:", NUM_STEPS)

In [ ]:
from pathlib import Path
import os
import subprocess

ROOT = Path("/content")
REPO = ROOT / "multi-source-diffusion-models"
MAMBA = ROOT / "bin" / "micromamba"
ENV_NAME = "msdm"

generation_code = r"""
from pathlib import Path
from typing import Callable, Optional
import json
import os
import torch
import torchaudio
import tqdm

from audio_diffusion_pytorch import KarrasSchedule
from main.module_base import Model

ROOT = Path(os.environ["REPO"])
CKPT_DIR = ROOT / "ckpts"

N_GENERATIONS = int(os.environ["N_GENERATIONS"])
GEN_BATCH_SIZE = int(os.environ["GEN_BATCH_SIZE"])
NUM_STEPS = int(os.environ["NUM_STEPS"])
SAMPLE_LEN = int(os.environ["SAMPLE_LEN"])
S_CHURN = float(os.environ["S_CHURN"])
SEED = int(os.environ["SEED"])

OUT_DIR = Path(os.environ["GEN_DIR"])
GEN_TRACKS_DIR = OUT_DIR / "gen_tracks"
GEN_MIX_DIR = OUT_DIR / "gen_mix"
GEN_ALL_DIR = OUT_DIR / "gen_all"

for path in [GEN_TRACKS_DIR, GEN_MIX_DIR, GEN_ALL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
SAMPLE_RATE = 22050
STEMS = ["bass", "drums", "guitar", "piano"]

def find_generation_ckpt():
    preferred = sorted(CKPT_DIR.rglob("glorious-star-335/*.ckpt"))
    if preferred:
        return preferred[0]

    all_ckpts = sorted(CKPT_DIR.rglob("*.ckpt"))
    if not all_ckpts:
        raise RuntimeError("No .ckpt found.")

    return all_ckpts[0]

def score_differential(x, sigma, denoise_fn):
    return (x - denoise_fn(x, sigma=sigma)) / sigma

@torch.no_grad()
def generate_track(
    denoise_fn: Callable,
    sigmas: torch.Tensor,
    noises: torch.Tensor,
    source: Optional[torch.Tensor] = None,
    mask: Optional[torch.Tensor] = None,
    num_resamples: int = 1,
    s_churn: float = 0.0,
    differential_fn: Callable = score_differential,
) -> torch.Tensor:
    x = sigmas[0] * noises
    source = torch.zeros_like(x) if source is None else source
    mask = torch.zeros_like(x) if mask is None else mask
    sigmas = sigmas.to(x.device)
    gamma = min(s_churn / (len(sigmas) - 1), 2**0.5 - 1)

    for i in tqdm.tqdm(range(len(sigmas) - 1)):
        sigma, sigma_next = sigmas[i], sigmas[i + 1]
        noisy_source = source + sigma * torch.randn_like(source)

        for r in range(num_resamples):
            x = mask * noisy_source + (1.0 - mask) * x
            sigma_hat = sigma * (gamma + 1)
            x_hat = x + torch.randn_like(x) * (sigma_hat**2 - sigma**2)**0.5
            d = differential_fn(x=x_hat, sigma=sigma_hat, denoise_fn=denoise_fn)
            x = x_hat + d * (sigma_next - sigma_hat)

            if r < num_resamples - 1:
                x = x + torch.randn_like(x) * (sigma**2 - sigma_next**2)**0.5

    return mask * source + (1.0 - mask) * x

def save_wav(path, wav):
    path.parent.mkdir(parents=True, exist_ok=True)
    wav = wav.detach().cpu().float()
    wav = torch.clamp(wav, -1.0, 1.0)
    torchaudio.save(str(path), wav, SAMPLE_RATE)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ckpt = find_generation_ckpt()

print("Using ckpt:", ckpt)
print("Device:", DEVICE)
print("N_GENERATIONS:", N_GENERATIONS)
print("GEN_BATCH_SIZE:", GEN_BATCH_SIZE)
print("NUM_STEPS:", NUM_STEPS)

model = Model.load_from_checkpoint(ckpt).to(DEVICE)
model.eval()

denoise_fn = model.model.diffusion.denoise_fn
schedule = KarrasSchedule(sigma_min=1e-4, sigma_max=20.0, rho=7)(NUM_STEPS, DEVICE)

meta = {
    "task": "MSDM total generation",
    "n_generations": N_GENERATIONS,
    "generation_batch_size": GEN_BATCH_SIZE,
    "num_steps": NUM_STEPS,
    "sample_len": SAMPLE_LEN,
    "sample_rate": SAMPLE_RATE,
    "s_churn": S_CHURN,
    "seed": SEED,
    "checkpoint": str(ckpt),
    "stems": STEMS,
}

with open(OUT_DIR / "generation_config.json", "w") as f:
    json.dump(meta, f, indent=2)

done = 0
while done < N_GENERATIONS:
    current_batch = min(GEN_BATCH_SIZE, N_GENERATIONS - done)
    print(f"Generating samples {done} to {done + current_batch - 1}")

    noises = torch.randn(current_batch, len(STEMS), SAMPLE_LEN, device=DEVICE)
    generated = generate_track(
        denoise_fn=denoise_fn,
        sigmas=schedule,
        noises=noises,
        s_churn=S_CHURN,
        num_resamples=1,
    )

    for i in range(current_batch):
        sample_id = done + i
        sample_name = f"Sample{sample_id:05d}"
        sample_dir = GEN_TRACKS_DIR / sample_name

        mix = generated[i].sum(dim=0, keepdim=True)
        save_wav(GEN_MIX_DIR / f"{sample_name}.wav", mix)

        for stem_idx, stem in enumerate(STEMS):
            stem_wav = generated[i, stem_idx:stem_idx + 1]
            save_wav(sample_dir / f"{stem}.wav", stem_wav)
            save_wav(GEN_ALL_DIR / f"{sample_name}_{stem}.wav", stem_wav)

    done += current_batch

print("Done generating.")
print("gen_tracks:", len(list(GEN_TRACKS_DIR.glob("Sample*"))))
print("gen_mix:", len(list(GEN_MIX_DIR.glob("*.wav"))))
print("gen_all:", len(list(GEN_ALL_DIR.glob("*.wav"))))
"""

env = os.environ.copy()
env["MAMBA_ROOT_PREFIX"] = str(ROOT / "micromamba")
env["PYTHONPATH"] = "."
env["REPO"] = str(REPO)
env["GEN_DIR"] = str(gen_dir)
env["N_GENERATIONS"] = str(N_GENERATIONS)
env["GEN_BATCH_SIZE"] = str(GEN_BATCH_SIZE)
env["NUM_STEPS"] = str(NUM_STEPS)
env["SAMPLE_LEN"] = str(SAMPLE_LEN)
env["S_CHURN"] = str(S_CHURN)
env["SEED"] = str(SEED)

msdm_lib = ROOT / "micromamba" / "envs" / ENV_NAME / "lib"
env["LD_LIBRARY_PATH"] = f"{msdm_lib}:{env.get('LD_LIBRARY_PATH', '')}"
env["LD_PRELOAD"] = str(msdm_lib / "libstdc++.so.6")

process = subprocess.Popen(
    [str(MAMBA), "run", "-n", ENV_NAME, "python", "-u", "-c", generation_code],
    cwd=str(REPO),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()
print("\nRETURN CODE:", return_code)
print("gen_dir:", gen_dir)

if return_code != 0:
    raise RuntimeError("Generation failed.")

In [ ]:
from pathlib import Path
import shutil
from google.colab import files

gen_dir = Path(gen_dir)

print("Using gen_dir:", gen_dir)
print("Exists:", gen_dir.exists())

if not gen_dir.exists():
    raise RuntimeError(f"Generation folder does not exist: {gen_dir}")

print("gen_tracks:", (gen_dir / "gen_tracks").exists(), len(list((gen_dir / "gen_tracks").glob("Sample*"))) if (gen_dir / "gen_tracks").exists() else 0)
print("gen_mix:", (gen_dir / "gen_mix").exists(), len(list((gen_dir / "gen_mix").glob("*.wav"))) if (gen_dir / "gen_mix").exists() else 0)
print("gen_all:", (gen_dir / "gen_all").exists(), len(list((gen_dir / "gen_all").glob("*.wav"))) if (gen_dir / "gen_all").exists() else 0)

bundle_dir = Path("/content/msdm_total_generation_samples")
zip_base = Path("/content/msdm_total_generation_samples")
zip_path = Path(str(zip_base) + ".zip")

if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
if zip_path.exists():
    zip_path.unlink()

bundle_dir.mkdir(parents=True, exist_ok=True)

folders_to_include = ["gen_tracks", "gen_mix", "gen_all"]
for folder_name in folders_to_include:
    src = gen_dir / folder_name
    dst = bundle_dir / folder_name

    if src.exists():
        shutil.copytree(src, dst)
        print("Copied folder:", src)
    else:
        print("Skipped missing folder:", src)

config_path = gen_dir / "generation_config.json"
if config_path.exists():
    shutil.copy2(config_path, bundle_dir / config_path.name)
    print("Copied file:", config_path)

if not (bundle_dir / "gen_tracks").exists() and not (bundle_dir / "gen_mix").exists():
    raise RuntimeError("No generated audio found.")

shutil.make_archive(
    base_name=str(zip_base),
    format="zip",
    root_dir=str(bundle_dir.parent),
    base_dir=bundle_dir.name,
)

print("\nCreated ZIP:", zip_path)
print("ZIP size GB:", round(zip_path.stat().st_size / 1024**3, 2))

files.download(str(zip_path))